# 움직임 시험 5 — 폭발만 그리기(배경 고정): ControlNet 윤곽 + LoRA로 검은 바탕에 이펙트만, 배경은 블로킹 배경을 고정 합성 (Kaggle·Colab T4, 학습 없음)
시험 4(docs/DECISIONS-2026-10-26.md): A 처음부터는 모양을 잘 따랐지만 건물·하늘이 장마다 바뀌었다 → 배경은 그리지 않는다.
- 블로킹에서 불·파편만 색으로 골라(따뜻한 색·흰 심) 검은 바탕 위에 둔 그림을 입력으로 쓴다. 배경은 블로킹의 장마다 불이 없는 픽셀로 만든 고정 한 장.
- **A 처음부터**: 강도 1.0 + 윤곽 조절 0.7(시험 4 A와 같음, 배경만 뺌).
- **C 색 시작**: 검은 바탕 불 블로킹을 시작점으로 강도 0.85 + 윤곽 0.6 — 위치·색 단계를 더 따르게.
- 합성: 고정 배경 위에 '스크린'(빛 더하기)으로 얹는다. 블로킹 불 둘레(약 40px 번짐) 밖에 생긴 것은 지운다.
- 같은 LoRA(3차 3바퀴)·시드(1234), 같은 블로킹 장은 결과 재사용.
- 설정(Kaggle): **GPU T4 x2**, **Internet On** → Save & Run All. 약 40–60분. Colab: T4 GPU → 모두 실행.
- 끝나면 `motion5_result.zip`(나란히 영상 `motion5.mp4` + 장별 그림 + 깜빡임 수치)을 Drive 공유 링크로 알려 준다.


In [ ]:
# 0) 어디서 돌리나: Kaggle이면 /kaggle/working, Colab이면 /content/out 에 결과를 쓴다
import os, glob
COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
OUT = "/content/out" if COLAB and not os.path.exists("/kaggle/working") else "/kaggle/working"
os.makedirs(OUT, exist_ok=True); print("결과 폴더", OUT)
# 1) 패키지와 자료: 기본 모델(HF), LoRA 3차 3바퀴·블로킹 두 가지(Drive)
# 버전 고정: Colab·Kaggle 기본 파이썬 3.13 + 새 transformers에서 diffusers 0.31이 깨졌다(FLAX_WEIGHTS_NAME, 2026-10-26). 서버 py3.13에서 아래 조합 import 확인
!pip install -q "diffusers==0.35.1" "transformers==4.56.2" "peft==0.17.1" "accelerate>=1.0" opencv-python-headless
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!curl -sSL -o /tmp/lora.safetensors "https://drive.usercontent.google.com/download?id=1ht_UqE8JkJTUX-eO5lNIqvhyFRcprUWJ&export=download&confirm=t"
import os, glob
for name, fid in [("num", "1lLCSZbiHE0QUO6xSw_uIeWiyUoNjlgDx"), ("clip", "1jITc3rXRduBnoW3C05I2XLmJ_7hEdtI6")]:
    os.system(f'curl -sSL -o /tmp/{name}.zip "https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t"')
    os.makedirs(f"/tmp/blk_{name}", exist_ok=True)
    os.system(f"cd /tmp/blk_{name} && unzip -q -o /tmp/{name}.zip '*.png'")
    print(name, len(glob.glob(f"/tmp/blk_{name}/*.png")), "장")
!ls -la /tmp/ilxl.safetensors /tmp/lora.safetensors


In [ ]:
# 2) 파이프라인: SDXL ControlNet 그림→그림 + LoRA 합치기 (윤곽 조절 xinsir/controlnet-canny-sdxl-1.0)
import torch
from diffusers import StableDiffusionXLControlNetImg2ImgPipeline, ControlNetModel, EulerAncestralDiscreteScheduler
from PIL import Image
cn = ControlNetModel.from_pretrained("xinsir/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_single_file("/tmp/ilxl.safetensors", controlnet=cn, torch_dtype=torch.float16)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights("/tmp/lora.safetensors", adapter_name="fx")
pipe.fuse_lora(lora_scale=1.0); pipe.unload_lora_weights()
pipe.to("cuda")
pipe.vae.to(torch.float32)
print(torch.cuda.memory_allocated() / 1e9, "GB")


In [ ]:
# 3) 폭발만 그리기 + 고정 배경 합성
import numpy as np, json, cv2
P = "nkmr_fx, vivid colors, anime screencap, explosion, fire, embers, white core orange rim, layered cel shading, no outline, sharp tips, large effect, no humans, black background, simple background"
N = "lowres, worst quality, text, watermark, city, building, scenery, sky, ground, horizon"
STEPS, CFG, SEED = 28, 6.0, 1234
def fxmask(b):  # 블로킹에서 이펙트(불·파편)만: 따뜻한 색 + 흰 심. 배경은 남색 하늘·검은 빌딩이라 색으로 갈린다(서버에서 확인)
    h = cv2.cvtColor(b, cv2.COLOR_RGB2HSV).astype(int)
    H, S, V = h[..., 0], h[..., 1], h[..., 2]
    return ((((H < 35) | (H > 165)) & (S > 70) & (V > 70)) | ((V > 200) & (S < 90))).astype(np.uint8)
def edges(a):
    e = cv2.Canny(cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), 30, 90)
    return Image.fromarray(np.stack([e] * 3, -1))
def run(init, ctrl, s, cs):
    g = torch.Generator("cuda").manual_seed(SEED)
    with torch.inference_mode():
        return pipe(prompt=P, negative_prompt=N, image=init, control_image=ctrl, strength=s, controlnet_conditioning_scale=cs,
                    num_inference_steps=STEPS, guidance_scale=CFG, generator=g, width=1344, height=768).images[0]
def screen(bg, fx, m):  # 고정 배경 위에 빛 더하기, 블로킹 불 둘레 밖은 지움
    f = np.asarray(fx, np.float32) / 255 * m[..., None]
    return Image.fromarray(((1 - (1 - bg / 255) * (1 - f)) * 255).clip(0, 255).astype(np.uint8))
def diff(a, b):
    return float(np.abs(np.asarray(a, np.float32) - np.asarray(b, np.float32)).mean())
log = {}
for name in ("num", "clip"):
    blk = [np.asarray(Image.open(p).convert("RGB")) for p in sorted(glob.glob(f"/tmp/blk_{name}/*.png"))]
    ms = [fxmask(b) for b in blk]
    st = np.stack(blk).astype(np.float32); mk = np.stack(ms)[..., None]
    cnt = (1 - mk).sum(0)
    bg = np.where(cnt > 0, (st * (1 - mk)).sum(0) / np.maximum(cnt, 1), 0)  # 불이 없던 장들의 평균 = 고정 배경
    hole = (cnt[..., 0] == 0).astype(np.uint8)
    if hole.any():
        bg = cv2.inpaint(bg.astype(np.uint8), hole, 5, cv2.INPAINT_TELEA).astype(np.float32)
    fxin = [b * m[..., None] for b, m in zip(blk, ms)]
    soft = [cv2.GaussianBlur(cv2.dilate(m, np.ones((41, 41), np.uint8)).astype(np.float32), (0, 0), 12) for m in ms]
    for tag, mkr in (("A_scratch", lambda a: (Image.fromarray(a), edges(a), 1.0, 0.7)), ("C_colorinit", lambda a: (Image.fromarray(a), edges(a), 0.85, 0.6))):
        cache, fxs, comps = {}, [], []
        for a, sm in zip(fxin, soft):
            key = a.tobytes()
            if key not in cache:
                cache[key] = run(*mkr(a)) if a.any() else Image.fromarray(np.zeros_like(a))
            fxs.append(cache[key]); comps.append(screen(bg, cache[key], sm))
        for sub, ims in ((tag, fxs), (tag + "_comp", comps)):
            d = f"{OUT}/m5/{name}/{sub}"; os.makedirs(d, exist_ok=True)
            for i, im in enumerate(ims):
                im.save(f"{d}/f{i:03d}.png")
            fl = np.mean([abs(diff(ims[i + 1], ims[i]) - diff(blk[i + 1], blk[i])) for i in range(len(ims) - 1)])
            log[f"{name} {sub}"] = round(float(fl), 2)
            print(name, sub, len(ims), "장, 깜빡임", log[f"{name} {sub}"])
    for sub, ims in (("blocking", blk), ("fx_in", fxin)):
        d0 = f"{OUT}/m5/{name}/{sub}"; os.makedirs(d0, exist_ok=True)
        for i, a in enumerate(ims):
            Image.fromarray(a.astype(np.uint8)).save(f"{d0}/f{i:03d}.png")
json.dump(log, open(f"{OUT}/motion5_flicker.json", "w"), ensure_ascii=False, indent=1)


In [ ]:
# 4) 나란히 영상: 위 줄 num, 아래 줄 clip / 칸 = 블로킹 | A 폭발만 | A 합성 | C 폭발만 | C 합성. 24fps, 3번 반복
rows = {n: {k: sorted(glob.glob(f"{OUT}/m5/{n}/{k}/*.png")) for k in ("blocking", "A_scratch", "A_scratch_comp", "C_colorinit", "C_colorinit_comp")} for n in ("num", "clip")}
def rd(p, label):
    im = cv2.resize(cv2.imread(p), (336, 192)); cv2.putText(im, label, (6, 20), 0, 0.55, (255, 255, 255), 2); return im
T = max(len(rows[n]["blocking"]) for n in rows)
vw = cv2.VideoWriter(f"{OUT}/motion5.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 24, (1680, 384))
for _ in range(3):
    for t in range(T):
        vw.write(np.vstack([np.hstack([rd(v[t % len(v)], f"{n} {k}") for k, v in rows[n].items()]) for n in rows]))
vw.release()
!cd {OUT} && zip -q -r motion5_result.zip m5 motion5.mp4 motion5_flicker.json && ls -la {OUT}


In [ ]:
# 5) Colab이면 결과 zip 내려받기(Kaggle은 Output 탭에서 받음)
if COLAB and OUT.startswith("/content"):
    from google.colab import files
    files.download(f"{OUT}/motion5_result.zip")
